In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load data
df = pd.read_csv("/content/cluster_labels_mapping.csv")

# 2. Load Texas shapefile
texas = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_5m.zip"
)
texas = texas[texas['STATEFP'] == '48']

# 3. Standardize names
texas['county'] = texas['NAME'].str.strip()
df['county'] = df['county'].str.replace(' County', '', case=False).str.strip()

# 4. Merge shapefile with data  ← the missing step
merged = texas.merge(df, on='county', how='left')
merged['cluster'] = merged['cluster'].astype(str)

# 5. Plot
fig, ax = plt.subplots(figsize=(14, 10))
merged.plot(
    column='cluster',
    ax=ax,
    cmap='tab10',
    legend=True,
    edgecolor='white',
    linewidth=0.4,
    missing_kwds={'color': 'lightgray', 'label': 'No data'},
    legend_kwds={'title': 'Cluster', 'loc': 'lower right'}
)
ax.set_title("Texas Counties by Cluster", fontsize=16, pad=12)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.patheffects as pe
from matplotlib.patches import Patch

merged['cluster'] = pd.to_numeric(merged['cluster'], errors='coerce').astype('Int64').astype(str)
merged['CODE2023'] = pd.to_numeric(merged['CODE2023'], errors='coerce').astype('Int64').astype(str)

NA = ('nan', '<NA>')

code_labels = {
    '1': 'Large central metro',
    '2': 'Large fringe metro',
    '3': 'Medium metro',
    '4': 'Small metro',
    '5': 'Micropolitan',
    '6': 'Noncore',
}

clusters = [str(c) for c in sorted(int(c) for c in merged['cluster'].unique() if c not in NA)]
codes = [str(c) for c in sorted(int(c) for c in merged['CODE2023'].unique() if c not in NA)]

cmap_names = ['Reds', 'Greens', 'Blues']
cluster_cmap = {c: matplotlib.colormaps[cmap_names[i]] for i, c in enumerate(clusters)}
shade = {c: 0.95 - 0.65 * i / (len(codes) - 1) for i, c in enumerate(codes)}

def blend(row):
    if row['cluster'] in NA or row['CODE2023'] in NA:
        return (0.83, 0.83, 0.83, 1)
    return cluster_cmap[row['cluster']](shade[row['CODE2023']])

merged['_clr'] = merged.apply(blend, axis=1)

fig, ax = plt.subplots(figsize=(11, 8))
merged.plot(color=merged['_clr'], ax=ax, edgecolor='white', linewidth=0.4)
ax.set_title('Cluster Map with Outbreak Counts', fontsize=15, fontweight='bold', pad=12)
ax.set_axis_off()

# --- outbreak points ---
outbreak_col = 'outbreak'
pts = merged[merged[outbreak_col].fillna(0) > 0].copy()
pts['_c'] = pts.geometry.representative_point()
pts['_x'] = pts['_c'].x
pts['_y'] = pts['_c'].y

vmax = pts[outbreak_col].max()
size = 80 + 1400 * (pts[outbreak_col] / vmax)

ax.scatter(pts['_x'], pts['_y'], s=size,
           facecolor='navajowhite', edgecolor='white', linewidth=0.8,
           alpha=0.75, zorder=5)

for _, r in pts.iterrows():
    ax.annotate(int(r[outbreak_col]), xy=(r['_x'], r['_y']),
                ha='center', va='center', fontsize=7, fontweight='bold',
                color='white', zorder=6,
                path_effects=[pe.withStroke(linewidth=1.5, foreground='black')])

# --- structured legend: grouped by cluster, NCHS labels ---
handles = []
for cl in clusters:
    handles.append(Patch(facecolor='none', edgecolor='none', label=f'Cluster {cl}'))
    for cd in codes:
        lbl = code_labels.get(cd, f'CODE {cd}')
        handles.append(Patch(facecolor=cluster_cmap[cl](shade[cd]),
                             edgecolor='gray', linewidth=0.4,
                             label=f'   {cd} · {lbl}'))
handles.append(Patch(facecolor=(0.83, 0.83, 0.83, 1),
                     edgecolor='gray', linewidth=0.4, label='No data'))

leg = ax.legend(handles=handles, title='Cluster · Urbanization',
                loc='center left', bbox_to_anchor=(1.0, 0.5),
                fontsize=11, title_fontsize=11, framealpha=0.95,
                handlelength=1.4, labelspacing=0.5)
leg.get_title().set_fontweight('bold')

plt.tight_layout()
plt.show()